# Tiong Bahru-Havelock SVI download: left and right views for each sample point

Based on `week9_SVI_download_colab.ipynb`. What changed:
- Input is `SVI_LeftRight_Manifest.csv`. Points and camera headings come from the manifest, so they are on the walking network.
- Each manifest row is one left or right view. A panorama shared by several points no longer skips a point.
- Parking aisle points not on an HDB-to-bus-stop route are removed (`points_on_hdb_bus_routes.csv`, `keep` column).
- Only Google-owned panoramas within 15 m are used. User uploads are excluded, because many are indoors.

**Two outputs, both in `svi_output/`:**
1. `svi_images/{image_id}.jpg`
2. `SVI_LeftRight_acquired.geojson`: one point per view with `image_status`, `image_path`, `capture_date`, `pano_id`. Points without an image stay in the file with their status.

**Run:** cells 1 to 5 with `TEST_N_POINTS = 3`, check the images, then set `TEST_N_POINTS = None` and run cells 1 to 5 again. Finished images are skipped and not billed again.

**Cost:** metadata has no charge. Each image counts against the Static Street View free allowance of 10,000 a month. `MAX_IMAGE_REQUESTS` stops the run.

**Before the full run:** confirm the Google terms question with the tutors. The key is never printed or written to outputs.

In [ ]:
# Cell 1 - Config (edit TEST_N_POINTS only)
import json, math, os, shutil, time
from pathlib import Path
import numpy as np, pandas as pd, requests

BASE = Path(r"G:\0-NUS\Sem1\AR5958A\Final Project\DATA\SVI scrap")
MANIFEST = BASE / "SVI_LeftRight_Manifest.csv"
ROUTE_FILTER = BASE / "source_screening" / "points_on_hdb_bus_routes.csv"
META_CACHE = BASE / "source_screening" / "google_metadata.csv"
OUT_DIR = BASE / "svi_output"
IMG_DIR = OUT_DIR / "svi_images"
OUT_GEOJSON = OUT_DIR / "SVI_LeftRight_acquired.geojson"
KEY_FILE = Path(r"G:\0-NUS\Sem1\AR5801\0-Week review\week9\scrap\scrapgoogle.api_key")

TEST_N_POINTS = 3            # first N eligible points; set None for all
MAX_IMAGE_REQUESTS = 5000    # billed requests per run; the full run needs at most about 4,300
MAX_DIST_M = 15.0            # sample point to panorama
IMAGE_SIZE = 640             # Static API maximum
REQUEST_TIMEOUT_SEC = 30
RETRIES = 3
SLEEP_SEC = 0.05
PROVIDER = "Google Street View Static API"

IMG_DIR.mkdir(parents=True, exist_ok=True)
API_KEY = os.environ.get("GOOGLE_SV_KEY") or KEY_FILE.read_text(encoding="utf-8-sig").strip()
assert API_KEY, "No API key"
print("key loaded:", bool(API_KEY), "| test points:", TEST_N_POINTS)

In [ ]:
# Cell 2 - API functions (exceptions report the type only: their text can contain the URL and key)
META_URL = "https://maps.googleapis.com/maps/api/streetview/metadata"
IMAGE_URL = "https://maps.googleapis.com/maps/api/streetview"
N_IMAGE_REQUESTS = 0

def get_with_retry(url, params):
    last = "no attempt"
    for k in range(RETRIES):
        try:
            r = requests.get(url, params={**params, "key": API_KEY}, timeout=REQUEST_TIMEOUT_SEC)
            if r.status_code in (429, 500, 502, 503):
                last = f"HTTP {r.status_code}"; time.sleep(2 * (k + 1)); continue
            return r
        except requests.RequestException as exc:
            last = type(exc).__name__; time.sleep(2 * (k + 1))
    raise RuntimeError(f"request failed: {last}")

def get_metadata(lat, lon):
    """Free. Returns the nearest outdoor panorama within 25 m, or its status."""
    j = get_with_retry(META_URL, {"location": f"{lat},{lon}", "radius": 25, "source": "outdoor"}).json()
    rec = dict(status=j.get("status", "UNKNOWN"), pano_id="", pano_lat=np.nan, pano_lon=np.nan, date="", copyright="")
    if rec["status"] == "OK":
        rec.update(pano_id=j["pano_id"], pano_lat=j["location"]["lat"], pano_lon=j["location"]["lng"], date=j.get("date", ""), copyright=j.get("copyright", ""))
    return rec

def download_view(pano_id, heading, pitch, fov, out_path):
    """One billed request."""
    global N_IMAGE_REQUESTS
    if N_IMAGE_REQUESTS >= MAX_IMAGE_REQUESTS:
        raise OverflowError(f"MAX_IMAGE_REQUESTS={MAX_IMAGE_REQUESTS} reached")
    N_IMAGE_REQUESTS += 1
    r = get_with_retry(IMAGE_URL, {"pano": pano_id, "size": f"{IMAGE_SIZE}x{IMAGE_SIZE}", "heading": round(float(heading), 2), "pitch": pitch, "fov": fov, "return_error_code": "true"})
    if r.status_code != 200 or not r.headers.get("content-type", "").startswith("image"):
        raise RuntimeError(f"image HTTP {r.status_code}")
    Path(out_path).write_bytes(r.content)

def distance_m(lat1, lon1, lat2, lon2):
    x = math.radians(lon2 - lon1) * math.cos(math.radians((lat1 + lat2) / 2))
    return 6371000 * math.hypot(x, math.radians(lat2 - lat1))

def write_geojson(df):
    features = []
    for r in df.to_dict("records"):
        props = {k: (None if isinstance(v, float) and math.isnan(v) else v) for k, v in r.items()}
        features.append({"type": "Feature", "geometry": {"type": "Point", "coordinates": [r["longitude"], r["latitude"]]}, "properties": props})
    tmp = OUT_GEOJSON.with_suffix(".tmp")
    tmp.write_text(json.dumps({"type": "FeatureCollection", "features": features}, default=str), encoding="utf-8")
    tmp.replace(OUT_GEOJSON)

In [ ]:
# Cell 3 - Build the view table: manifest rows, route filter, panorama match (metadata only)
views = pd.read_csv(MANIFEST, encoding="utf-8-sig", dtype={"image_provider": str, "image_path": str, "capture_date": str})
keep = pd.read_csv(ROUTE_FILTER, encoding="utf-8-sig")[["sample_id", "keep", "road_group", "osm_class", "osm_service"]]
views = views.merge(keep, on="sample_id", how="left")
n_before = views.sample_id.nunique()
views = views[views.keep.fillna(True).astype(bool)].drop(columns="keep").reset_index(drop=True)
print(f"points {n_before} -> {views.sample_id.nunique()} after removing parking aisles not on routes; views {len(views)}")

cache = pd.read_csv(META_CACHE, encoding="utf-8-sig") if META_CACHE.exists() else pd.DataFrame(columns=["sample_id", "status", "pano_id", "pano_lat", "pano_lon", "date", "copyright"])
pts = views.drop_duplicates("sample_id")
todo = pts[~pts.sample_id.isin(cache.sample_id)]
new = []
for r in todo.itertuples():
    new.append({"sample_id": r.sample_id, **get_metadata(r.latitude, r.longitude)})
if new:
    cache = pd.concat([cache, pd.DataFrame(new)], ignore_index=True)
print("metadata queried now:", len(new), "| from cache:", len(pts) - len(new))

m = cache[["sample_id", "status", "pano_id", "pano_lat", "pano_lon", "date", "copyright"]].drop_duplicates("sample_id")
views = views.merge(m, on="sample_id", how="left")
views["match_distance_m"] = [round(distance_m(a, b, c, d), 1) if c == c else np.nan for a, b, c, d in zip(views.latitude, views.longitude, views.pano_lat, views.pano_lon)]
views["is_google_owned"] = views.copyright.fillna("").str.contains("Google")
views["eligible"] = (views.status == "OK") & views.is_google_owned & (views.match_distance_m <= MAX_DIST_M)
views["image_provider"] = np.where(views.eligible, PROVIDER, "")
views["image_path"] = ""
views["capture_date"] = views.date.where(views.eligible, "").fillna("")
views["image_status"] = np.where(views.eligible, "not_acquired", "no_google_pano")
print(views.image_status.value_counts().to_dict())

In [ ]:
# Cell 4 - Download the left and right views; update the GeoJSON as it goes
targets = views[views.eligible]
if TEST_N_POINTS:
    targets = targets[targets.sample_id.isin(targets.sample_id.drop_duplicates().head(TEST_N_POINTS))]
print(f"views to process: {len(targets)} (points {targets.sample_id.nunique()})")

same_view = {}   # identical panorama and heading: copy the file, no second request
n_copied = n_existing = n_failed = 0
stop = False
for i, r in enumerate(targets.itertuples()):
    path = IMG_DIR / f"{r.image_id}.jpg"
    key = (r.pano_id, round(float(r.camera_heading_deg)))
    try:
        if path.exists() and path.stat().st_size > 0:
            n_existing += 1
        elif key in same_view:
            shutil.copyfile(same_view[key], path); n_copied += 1
        else:
            download_view(r.pano_id, r.camera_heading_deg, r.camera_pitch_deg, r.camera_fov_deg, path)
            time.sleep(SLEEP_SEC)
        same_view.setdefault(key, path)
    except OverflowError as exc:
        print(exc); stop = True; break
    except Exception as exc:
        n_failed += 1
        views.loc[r.Index, "image_status"] = "failed"
        print(r.image_id, exc)
        if n_failed >= 20:
            print("Stopping: 20 failures"); stop = True; break
        continue
    views.loc[r.Index, ["image_status", "image_path"]] = ["acquired", f"svi_images/{path.name}"]
    if (i + 1) % 200 == 0:
        write_geojson(views.drop(columns=["eligible", "status", "date", "copyright"]))
        print(f"  {i + 1}/{len(targets)} billed {N_IMAGE_REQUESTS}", flush=True)

# Images from an earlier run also count, whatever TEST_N_POINTS is set to now
for r in views[views.eligible & (views.image_status == "not_acquired")].itertuples():
    path = IMG_DIR / f"{r.image_id}.jpg"
    if path.exists() and path.stat().st_size > 0:
        views.loc[r.Index, ["image_status", "image_path"]] = ["acquired", f"svi_images/{path.name}"]
write_geojson(views.drop(columns=["eligible", "status", "date", "copyright"]))
print(f"billed requests this run: {N_IMAGE_REQUESTS} | already on disk: {n_existing} | copied: {n_copied} | failed: {n_failed} | stopped early: {stop}")

In [ ]:
# Cell 5 - Check that the two outputs match
gj = json.loads(OUT_GEOJSON.read_text(encoding="utf-8"))
g = pd.DataFrame([f["properties"] for f in gj["features"]])
print("geojson views:", len(g), "| status:", g.image_status.value_counts().to_dict())
acq = g[g.image_status == "acquired"]
missing = [p for p in acq.image_path if not (OUT_DIR / p).exists()]
files = {p.name for p in IMG_DIR.glob("*.jpg")}
extra = files - {Path(p).name for p in acq.image_path}
print("acquired rows:", len(acq), "| files:", len(files), "| rows without file:", len(missing), "| files without row:", len(extra))
sides = acq.groupby("sample_id").side.nunique()
print("points with both L and R:", int((sides == 2).sum()), "| with one side only:", int((sides == 1).sum()))
print("points with no image:", int((g.groupby("sample_id").image_status.apply(lambda s: (s != "acquired").all())).sum()))
assert not missing and not extra, "outputs do not match"
if len(acq):
    from PIL import Image
    try:
        from IPython.display import display
    except ImportError:
        display = None
    first = acq.sample_id.iloc[0]
    strip = Image.new("RGB", (2 * IMAGE_SIZE, IMAGE_SIZE))
    for k, s in enumerate(["L", "R"]):
        strip.paste(Image.open(IMG_DIR / f"{first}_{s}.jpg"), (k * IMAGE_SIZE, 0))
    print(first, ": left | right")
    if display:
        display(strip)